### Prompt Manager Test

Day 4 작업 중 프롬프트 관리 모듈 기능 테스트

테스트 대상:
- PromptManager: 프롬프트 로드 및 관리
- 프롬프트 템플릿 변수 치환
- LLM 메시지 빌드
- 카테고리별 프롬프트 접근

In [1]:
import json
import sys
from pathlib import Path

from dotenv import load_dotenv

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Load environment variables
load_dotenv(project_root / ".env")

from app.core.prompts import PromptManager, get_prompt_manager, build_messages, format_prompt

print("✓ Setup complete")

✓ Setup complete


### 1. PromptManager Initialization

PromptManager 인스턴스 생성 및 기본 정보 확인

In [2]:
# PromptManager 인스턴스 생성
manager = PromptManager()

print("PromptManager Initialization Test:")
print("=" * 60)
print(f"Prompts file path: {manager.prompts_path}")
print(f"File exists: {manager.prompts_path.exists()}")
print(f"Prompts loaded: {manager._prompts is not None}")

PromptManager Initialization Test:
Prompts file path: /mnt/d/project/research-curator/configs/prompts.yaml
File exists: True
Prompts loaded: False


- 실제에서는 싱글턴 인스턴스 반환을 위해 `get_prompt_manager` 사용

### 2. Available Categories

사용 가능한 프롬프트 카테고리 목록 확인

In [3]:
# 카테고리 목록 확인
categories = manager.get_categories()

print("Available Prompt Categories:")
print("=" * 60)
for i, category in enumerate(categories, 1):
    print(f"{i}. {category}")

print(f"\nTotal categories: {len(categories)}")

Available Prompt Categories:
1. summarize
2. evaluate_importance
3. classify_category
4. extract_metadata
5. onboarding
6. common

Total categories: 6


### 3. Summary Length Options

요약 길이 옵션 확인

In [4]:
# 요약 길이 옵션
summary_lengths = manager.get_summary_lengths()

print("Summary Length Options:")
print("=" * 60)
for length in summary_lengths:
    print(f"  - {length}")

print(f"\nTotal options: {len(summary_lengths)}")

Summary Length Options:
  - short
  - medium
  - long

Total options: 3


### 4. Get Specific Prompt

특정 프롬프트 가져오기 테스트

In [5]:
# 특정 프롬프트 가져오기
system_prompt = manager.get("summarize.korean.medium.system")

print("Get Specific Prompt Test:")
print("=" * 60)
print("Path: summarize.korean.medium.system")
print("\nSystem Prompt:")
print("-" * 60)
print(system_prompt)
print("-" * 60)
print(f"\nPrompt length: {len(system_prompt)} characters")

Get Specific Prompt Test:
Path: summarize.korean.medium.system

System Prompt:
------------------------------------------------------------
당신은 AI 연구 분야의 전문 리서처입니다.
주어진 논문이나 기사를 정확하고 명확하게 요약하는 것이 당신의 임무입니다.

요약 시 다음 원칙을 따르세요:
- 핵심 아이디어와 주요 발견을 포함
- 기술적 세부사항은 적절히 간략화
- 전문 용어는 유지하되 이해하기 쉽게 설명
- 논리적 흐름 유지
- 객관적이고 중립적인 톤 유지

------------------------------------------------------------

Prompt length: 182 characters


### 5. Get User Template

유저 프롬프트 템플릿 가져오기

In [6]:
# 유저 템플릿 가져오기
user_template = manager.get_user_template("summarize", "korean.medium")

print("User Template Test:")
print("=" * 60)
print("Category: summarize")
print("Subcategory: korean.medium")
print("\nUser Template:")
print("-" * 60)
print(user_template)
print("-" * 60)
print(f"\nTemplate variables: {user_template.count('{')} found")

# 템플릿 변수 추출
import re
variables = re.findall(r'{(\w+)}', user_template)
print(f"Variables: {variables}")

User Template Test:
Category: summarize
Subcategory: korean.medium

User Template:
------------------------------------------------------------
다음 내용을 3-5문장으로 요약해주세요. 핵심 아이디어와 주요 발견을 명확히 전달하세요.

제목: {title}

내용:
{content}

요약:

------------------------------------------------------------

Template variables: 2 found
Variables: ['title', 'content']


### 6. Format Prompt with Variables

프롬프트 템플릿에 변수 치환하기

In [7]:
# 프롬프트 포맷팅
template = "제목: {title}\n내용: {content}\n저자: {author}"

formatted = manager.format_prompt(
    template,
    title="Attention Is All You Need",
    content="Transformer architecture를 소개하는 논문...",
    author="Vaswani et al."
)

print("Prompt Formatting Test:")
print("=" * 60)
print("Original Template:")
print(template)
print("\nFormatted Result:")
print("-" * 60)
print(formatted)
print("-" * 60)

Prompt Formatting Test:
Original Template:
제목: {title}
내용: {content}
저자: {author}

Formatted Result:
------------------------------------------------------------
제목: Attention Is All You Need
내용: Transformer architecture를 소개하는 논문...
저자: Vaswani et al.
------------------------------------------------------------


### 7. Build Messages for LLM

LLM API용 메시지 리스트 생성

In [8]:
# 메시지 빌드
messages = manager.build_messages(
    "summarize",
    "korean.medium",
    title="GPT-4 Technical Report",
    content="""GPT-4는 OpenAI가 개발한 대규모 멀티모달 모델입니다. 
    이미지와 텍스트 입력을 받아 텍스트를 생성할 수 있으며, 
    다양한 전문적이고 학술적인 벤치마크에서 인간 수준의 성능을 보입니다."""
)

print("Build Messages Test:")
print("=" * 60)
print(f"Total messages: {len(messages)}\n")

for i, msg in enumerate(messages, 1):
    print(f"Message {i}: {msg['role']}")
    print("-" * 60)
    print(msg['content'][:200] + "..." if len(msg['content']) > 200 else msg['content'])
    print()

Build Messages Test:
Total messages: 2

Message 1: system
------------------------------------------------------------
당신은 AI 연구 분야의 전문 리서처입니다.
주어진 논문이나 기사를 정확하고 명확하게 요약하는 것이 당신의 임무입니다.

요약 시 다음 원칙을 따르세요:
- 핵심 아이디어와 주요 발견을 포함
- 기술적 세부사항은 적절히 간략화
- 전문 용어는 유지하되 이해하기 쉽게 설명
- 논리적 흐름 유지
- 객관적이고 중립적인 톤 유지


Message 2: user
------------------------------------------------------------
다음 내용을 3-5문장으로 요약해주세요. 핵심 아이디어와 주요 발견을 명확히 전달하세요.

제목: GPT-4 Technical Report

내용:
GPT-4는 OpenAI가 개발한 대규모 멀티모달 모델입니다. 
    이미지와 텍스트 입력을 받아 텍스트를 생성할 수 있으며, 
    다양한 전문적이고 학술적인 벤치마크에서 인간 수준의 성능을 보입니다.

...



### 8. Test All Summary Lengths

모든 요약 길이 옵션에 대한 메시지 생성

In [9]:
# 모든 요약 길이로 메시지 생성
test_content = {
    "title": "Transformer Neural Networks",
    "content": "A comprehensive overview of transformer architecture and its applications."
}

print("Summary Length Comparison:")
print("=" * 60)

for length in summary_lengths:
    messages = manager.build_messages(
        "summarize",
        f"korean.{length}",
        **test_content
    )
    user_msg = messages[1]['content']
    
    print(f"\nLength: {length}")
    print(f"User prompt preview: {user_msg[:100]}...")
    print(f"Total length: {len(user_msg)} chars")

Summary Length Comparison:

Length: short
User prompt preview: 다음 내용을 2-3문장으로 간단히 요약해주세요.

제목: Transformer Neural Networks

내용:
A comprehensive overview of transfo...
Total length: 145 chars

Length: medium
User prompt preview: 다음 내용을 3-5문장으로 요약해주세요. 핵심 아이디어와 주요 발견을 명확히 전달하세요.

제목: Transformer Neural Networks

내용:
A comprehens...
Total length: 168 chars

Length: long
User prompt preview: 다음 내용을 6-8문장으로 상세히 요약해주세요. 배경, 방법론, 결과, 의미를 포함하세요.

제목: Transformer Neural Networks

내용:
A comprehen...
Total length: 169 chars


### 9. Importance Evaluation Prompt

중요도 평가 프롬프트 테스트

In [10]:
# 중요도 평가 프롬프트
importance_messages = manager.build_messages(
    "evaluate_importance",
    title="GPT-5 Achieves Human-Level AGI",
    content="OpenAI announces GPT-5 with breakthrough capabilities...",
    metadata="{\"source\": \"openai.com\", \"date\": \"2024-12-14\"}"
)

print("Importance Evaluation Prompt Test:")
print("=" * 60)

print("System Prompt:")
print("-" * 60)
print(importance_messages[0]['content'][:300] + "...")

print("\nUser Prompt:")
print("-" * 60)
print(importance_messages[1]['content'])

# 평가 기준 및 가중치
print("\n" + "=" * 60)
print("Evaluation Criteria:")
criteria = manager.get_evaluation_criteria()
weights = manager.get_evaluation_weights()

for criterion in criteria:
    weight = weights.get(criterion, 0)
    print(f"  - {criterion}: {weight * 100}%")

Importance Evaluation Prompt Test:
System Prompt:
------------------------------------------------------------
당신은 AI 연구 동향을 분석하는 전문가입니다.
주어진 논문이나 기사의 중요도를 객관적으로 평가하는 것이 당신의 임무입니다.

다음 4가지 기준으로 평가하세요:

1. **혁신성 (Innovation)**: 0.0 ~ 1.0
   - 새로운 아이디어나 접근법을 제시하는가?
   - 기존 방법론을 획기적으로 개선했는가?
   - 독창적인가?

2. **관련성 (Relevance)**: 0.0 ~ 1.0
   - AI 연구 분야에서 얼마나 관련성이 높은가?
   - 현재 트렌드와 연결되는가?
   - 실용적 가치가 있는가?

3. **...

User Prompt:
------------------------------------------------------------
다음 내용의 중요도를 평가하세요.

제목: GPT-5 Achieves Human-Level AGI

내용:
OpenAI announces GPT-5 with breakthrough capabilities...

메타데이터:
{"source": "openai.com", "date": "2024-12-14"}

JSON 형식으로 응답하세요:
${${
  "innovation": 0.0-1.0,
  "relevance": 0.0-1.0,
  "impact": 0.0-1.0,
  "timeliness": 0.0-1.0,
  "reasoning": "평가 근거 설명",
  "overall_score": 0.0-1.0
}}


Evaluation Criteria:
  - innovation: 30.0%
  - relevance: 25.0%
  - impact: 30.0%
  - timeliness: 15.0%


### 10. Category Classification Prompt

카테고리 분류 프롬프트 테스트

In [11]:
# 카테고리 분류 프롬프트
classification_messages = manager.build_messages(
    "classify_category",
    title="Attention Is All You Need",
    content="We propose a new simple network architecture, the Transformer...",
    source_name="arXiv",
    url="https://arxiv.org/abs/1706.03762"
)

print("Category Classification Prompt Test:")
print("=" * 60)

print("System Prompt:")
print("-" * 60)
print(classification_messages[0]['content'][:300] + "...")

print("\nUser Prompt:")
print("-" * 60)
print(classification_messages[1]['content'])

# 사용 가능한 카테고리 및 연구 분야
print("\n" + "=" * 60)
print("Available Categories:")
categories = manager.get_classification_categories()
for cat in categories:
    print(f"  - {cat}")

print("\nResearch Fields:")
fields = manager.get_research_fields()
for field in fields:
    print(f"  - {field}")

Category Classification Prompt Test:
System Prompt:
------------------------------------------------------------
당신은 콘텐츠 분류 전문가입니다.
주어진 내용을 정확한 카테고리로 분류하고, 추가 메타데이터를 추출하는 것이 당신의 임무입니다.

카테고리 정의:
- **paper**: 학술 논문 (arXiv, 학회, 저널 등)
- **news**: 뉴스 기사 (언론사, 테크 블로그 등)
- **report**: 연구 리포트 (기업, 연구소 보고서)
- **blog**: 개인 블로그 포스트 또는 기술 블로그
- **other**: 기타

추가로 다음 정보를 추출하세요:
- 주요 키워드 (3-5개)
- 연구 분야 (Machine Learning, N...

User Prompt:
------------------------------------------------------------
다음 내용을 분류하고 메타데이터를 추출하세요.

제목: Attention Is All You Need

내용:
We propose a new simple network architecture, the Transformer...

출처: arXiv
URL: https://arxiv.org/abs/1706.03762

JSON 형식으로 응답하세요:
${${
  "category": "paper|news|report|blog|other",
  "confidence": 0.0-1.0,
  "keywords": ["keyword1", "keyword2", ...],
  "research_field": "주요 연구 분야",
  "sub_fields": ["세부 분야1", "세부 분야2"],
  "reasoning": "분류 근거"
}}


Available Categories:
  - paper
  - news
  - report
  - blog
  - other

Research Fields:
  - Ma

### 11. Metadata Extraction Prompt

메타데이터 추출 프롬프트 테스트

In [12]:
# 메타데이터 추출 프롬프트
metadata_messages = manager.build_messages(
    "extract_metadata",
    title="BERT: Pre-training of Deep Bidirectional Transformers",
    content="""Jacob Devlin, Ming-Wei Chang, Kenton Lee, Kristina Toutanova
    Google AI Language
    We introduce BERT, which achieves state-of-the-art results on 11 NLP tasks.
    Code available at: github.com/google-research/bert
    """
)

print("Metadata Extraction Prompt Test:")
print("=" * 60)

print("System Prompt:")
print("-" * 60)
print(metadata_messages[0]['content'])

print("\nUser Prompt:")
print("-" * 60)
print(metadata_messages[1]['content'])

Metadata Extraction Prompt Test:
System Prompt:
------------------------------------------------------------
당신은 콘텐츠에서 구조화된 메타데이터를 추출하는 전문가입니다.
주어진 내용에서 유용한 정보를 파악하고 JSON 형식으로 정리하는 것이 당신의 임무입니다.

추출할 정보:
- 저자 정보 (있는 경우)
- 소속 기관
- 발행일
- 주요 기술/방법론
- 데이터셋 (사용된 경우)
- 성능 지표 (있는 경우)
- 관련 링크 (GitHub, Demo 등)


User Prompt:
------------------------------------------------------------
다음 내용에서 메타데이터를 추출하세요.

제목: BERT: Pre-training of Deep Bidirectional Transformers

내용:
Jacob Devlin, Ming-Wei Chang, Kenton Lee, Kristina Toutanova
    Google AI Language
    We introduce BERT, which achieves state-of-the-art results on 11 NLP tasks.
    Code available at: github.com/google-research/bert
    

JSON 형식으로 응답하세요:
${${
  "authors": ["저자1", "저자2"],
  "affiliations": ["기관1", "기관2"],
  "publication_date": "YYYY-MM-DD",
  "technologies": ["기술1", "기술2"],
  "datasets": ["데이터셋1", "데이터셋2"],
  "metrics": ${${"metric_name": "value"}},
  "links": ${${"github": "url", "demo": "url"}},
  "references": ["중요 참고문헌"]
}

### 12. Onboarding Prompts

온보딩 챗봇 프롬프트 테스트

In [13]:
# 온보딩 프롬프트
onboarding_data = manager.get("onboarding")

print("Onboarding Chatbot Prompts:")
print("=" * 60)

print("System Prompt:")
print("-" * 60)
print(onboarding_data['system'])

print("\nWelcome Message:")
print("-" * 60)
print(onboarding_data['welcome_message'])

print("\nQuestions:")
print("=" * 60)
questions = onboarding_data['questions']
for key, value in questions.items():
    print(f"\n{key}:")
    print(f"  Prompt: {value['prompt'][:100]}...")
    if 'examples' in value:
        print(f"  Examples: {value['examples']}")
    if 'default' in value:
        print(f"  Default: {value['default']}")

Onboarding Chatbot Prompts:
System Prompt:
------------------------------------------------------------
당신은 친근하고 전문적인 AI 리서치 큐레이션 서비스의 온보딩 어시스턴트입니다.

당신의 역할:
- 사용자의 연구 분야와 관심사를 자연스럽게 파악
- 적절한 서비스 설정 추천
- 친근하면서도 전문적인 톤 유지
- 명확하고 간결한 질문

수집할 정보:
1. 연구 분야 (예: Machine Learning, NLP, Computer Vision)
2. 주요 관심 키워드 (예: GPT, Transformer, Diffusion Models)
3. 선호하는 정보 유형 (논문/뉴스/리포트 비중)
4. 추가하고 싶은 소스 사이트
5. 이메일 발송 시간
6. 하루 수신 희망 개수

대화는 자연스럽게 진행하되, 필요한 정보를 빠짐없이 수집하세요.


Welcome Message:
------------------------------------------------------------
안녕하세요! 👋

AI 리서치 큐레이션 서비스에 오신 것을 환영합니다.
매일 아침, 당신이 관심 있는 최신 AI 연구 동향을 선별하여 이메일로 보내드립니다.

맞춤형 큐레이션을 위해 몇 가지 질문을 드리겠습니다.
시작할까요?


Questions:

research_field:
  Prompt: 주로 어떤 연구 분야에 관심이 있으신가요? (예: Machine Learning, NLP, Computer Vision)...
  Examples: ['Machine Learning', 'Natural Language Processing', 'Computer Vision', 'Reinforcement Learning']

keywords:
  Prompt: 특별히 관심 있는 주제나 키워드가 있나요? (예: GPT, Transformer, Diffusion Models)...
  Examples: ['Large Lang

### 13. Common Settings

공통 설정 확인

In [14]:
# 공통 설정
common = manager.get("common")

print("Common Settings:")
print("=" * 60)

print(f"Default language: {common['default_language']}")
print(f"Default summary length: {common['default_summary_length']}")

print("\nJSON Instruction:")
print("-" * 60)
print(common['json_instruction'])

print("\nError Messages:")
print("-" * 60)
for key, msg in common['error_messages'].items():
    print(f"  {key}: {msg}")

Common Settings:
Default language: ko
Default summary length: medium

JSON Instruction:
------------------------------------------------------------

IMPORTANT: You must respond with valid JSON only. Do not include any explanation or text outside the JSON structure.


Error Messages:
------------------------------------------------------------
  invalid_json: LLM이 유효한 JSON을 반환하지 않았습니다.
  empty_response: LLM 응답이 비어있습니다.
  rate_limit: API rate limit에 도달했습니다. 잠시 후 다시 시도하세요.
  api_error: LLM API 호출 중 오류가 발생했습니다.


### 14. Singleton Pattern Test

get_prompt_manager() 싱글톤 패턴 테스트

In [15]:
# 싱글톤 패턴 테스트
manager1 = get_prompt_manager()
manager2 = get_prompt_manager()

print("Singleton Pattern Test:")
print("=" * 60)
print(f"Manager 1 ID: {id(manager1)}")
print(f"Manager 2 ID: {id(manager2)}")
print(f"Are they the same instance? {manager1 is manager2}")
print(f"\n✓ Singleton pattern working correctly" if manager1 is manager2 else "✗ Singleton pattern failed")

Singleton Pattern Test:
Manager 1 ID: 140596582796608
Manager 2 ID: 140596582796608
Are they the same instance? True

✓ Singleton pattern working correctly


### 15. Convenience Functions Test

편의 함수 테스트

In [16]:
# 편의 함수 테스트
from app.core.prompts import get_prompt, build_messages, format_prompt

print("Convenience Functions Test:")
print("=" * 60)

# 1. get_prompt
prompt = get_prompt("summarize.korean.short.system")
print("1. get_prompt():")
print(f"   Retrieved: {prompt[:50]}...")

# 2. build_messages
messages = build_messages(
    "summarize",
    "korean.short",
    title="Test",
    content="Content"
)
print(f"\n2. build_messages():")
print(f"   Generated {len(messages)} messages")

# 3. format_prompt
template = "Hello {name}, welcome to {place}"
formatted = format_prompt(template, name="Alice", place="Wonderland")
print(f"\n3. format_prompt():")
print(f"   Result: {formatted}")

Convenience Functions Test:
1. get_prompt():
   Retrieved: 당신은 AI 연구 분야의 전문 리서처입니다.
주어진 논문이나 기사를 정확하고 간결하게 요약...

2. build_messages():
   Generated 2 messages

3. format_prompt():
   Result: Hello Alice, welcome to Wonderland


### 16. Prompt Reload Test

프롬프트 재로드 기능 테스트

In [17]:
# 재로드 테스트
manager = get_prompt_manager()

print("Prompt Reload Test:")
print("=" * 60)

# 초기 상태
initial_categories = manager.get_categories()
print(f"Initial categories count: {len(initial_categories)}")

# 재로드
manager.reload()
print("Prompts reloaded")

# 재로드 후 상태
reloaded_categories = manager.get_categories()
print(f"Reloaded categories count: {len(reloaded_categories)}")

print(f"\n✓ Reload successful" if len(initial_categories) == len(reloaded_categories) else "✗ Reload failed")

Prompt Reload Test:
Initial categories count: 6
Prompts reloaded
Reloaded categories count: 6

✓ Reload successful


### 17. Missing Key Handling

존재하지 않는 키 처리 테스트

In [18]:
# 존재하지 않는 키 테스트
print("Missing Key Handling Test:")
print("=" * 60)

# 1. get() with default
result1 = manager.get("nonexistent.key", default="DEFAULT_VALUE")
print(f"1. get('nonexistent.key', default='DEFAULT_VALUE'):")
print(f"   Result: {result1}")

# 2. get() without default
result2 = manager.get("another.missing.key")
print(f"\n2. get('another.missing.key'):")
print(f"   Result: {result2}")

# 3. get_system_prompt() with invalid category
result3 = manager.get_system_prompt("invalid_category")
print(f"\n3. get_system_prompt('invalid_category'):")
print(f"   Result: {result3}")

# 4. build_messages() with invalid category (should raise error)
print(f"\n4. build_messages('invalid_category'):")
try:
    manager.build_messages("invalid_category", title="Test", content="Test")
    print("   ✗ Should have raised ValueError")
except ValueError as e:
    print(f"   ✓ Correctly raised ValueError: {e}")

Missing Key Handling Test:
1. get('nonexistent.key', default='DEFAULT_VALUE'):
   Result: DEFAULT_VALUE

2. get('another.missing.key'):
   Result: None

3. get_system_prompt('invalid_category'):
   Result: None

4. build_messages('invalid_category'):
   ✓ Correctly raised ValueError: Prompts not found for invalid_category


### 18. Integration Test with LLM

실제 LLM API와 통합 테스트

In [19]:
from app.llm import LLMClient

# LLM 클라이언트 생성
llm_client = LLMClient(provider="openai", temperature=0.5)

# 테스트 데이터
test_article = {
    "title": "GPT-4 Technical Report",
    "content": """GPT-4 is a large multimodal model that can accept image and text inputs 
    and produce text outputs. It exhibits human-level performance on various professional 
    and academic benchmarks. GPT-4 is more reliable, creative, and able to handle much 
    more nuanced instructions than GPT-3.5."""
}

print("Integration Test with LLM:")
print("=" * 60)

# 1. 요약 생성 (short)
print("\n1. Korean Short Summary:")
print("-" * 60)
messages = manager.build_messages(
    "summarize",
    "korean.short",
    **test_article
)
summary_short = llm_client.chat_completion(messages, max_tokens=200)
print(summary_short)

# 2. 요약 생성 (medium)
print("\n2. Korean Medium Summary:")
print("-" * 60)
messages = manager.build_messages(
    "summarize",
    "korean.medium",
    **test_article
)
summary_medium = llm_client.chat_completion(messages, max_tokens=300)
print(summary_medium)

print("\n" + "=" * 60)
print(f"Short summary length: {len(summary_short)} chars")
print(f"Medium summary length: {len(summary_medium)} chars")

Integration Test with LLM:

1. Korean Short Summary:
------------------------------------------------------------
GPT-4는 이미지와 텍스트 입력을 받아 텍스트 출력이 가능한 대규모 멀티모달 모델로, 다양한 전문 및 학문적 기준에서 인간 수준의 성능을 보입니다. GPT-3.5보다 더 신뢰할 수 있고 창의적이며, 세부적인 지시를 처리하는 능력이 향상되었습니다.

2. Korean Medium Summary:
------------------------------------------------------------
GPT-4는 이미지와 텍스트 입력을 받아 텍스트 출력을 생성할 수 있는 대규모 멀티모달 모델입니다. 이 모델은 다양한 전문 및 학술 기준에서 인간 수준의 성능을 보이며, GPT-3.5보다 더 신뢰할 수 있고 창의적이며, 복잡한 지시를 처리하는 능력이 향상되었습니다.

Short summary length: 138 chars
Medium summary length: 148 chars


### 19. Importance Evaluation with LLM

중요도 평가 프롬프트 + LLM 통합 테스트

In [20]:
# 중요도 평가 테스트
evaluation_data = {
    "title": "AlphaFold 3 Predicts Protein-Ligand Structures",
    "content": """AlphaFold 3 extends the capabilities of AlphaFold 2 by predicting 
    protein-ligand complex structures with unprecedented accuracy. This breakthrough 
    has significant implications for drug discovery and molecular biology.""",
    "metadata": json.dumps({"source": "Nature", "date": "2024-12-01"})
}

print("Importance Evaluation with LLM:")
print("=" * 60)

messages = manager.build_messages(
    "evaluate_importance",
    **evaluation_data
)

evaluation_result = llm_client.chat_completion(
    messages,
    response_format="json",
    max_tokens=400
)

print("Evaluation Result:")
print("-" * 60)
print(evaluation_result)

# JSON 파싱
try:
    eval_data = json.loads(evaluation_result)
    print("\nParsed Evaluation:")
    print("-" * 60)
    print(f"Innovation: {eval_data.get('innovation', 'N/A')}")
    print(f"Relevance: {eval_data.get('relevance', 'N/A')}")
    print(f"Impact: {eval_data.get('impact', 'N/A')}")
    print(f"Timeliness: {eval_data.get('timeliness', 'N/A')}")
    print(f"Overall Score: {eval_data.get('overall_score', 'N/A')}")
    print(f"\nReasoning: {eval_data.get('reasoning', 'N/A')}")
except json.JSONDecodeError as e:
    print(f"✗ JSON parsing failed: {e}")

Importance Evaluation with LLM:
Evaluation Result:
------------------------------------------------------------
{
  "innovation": 0.9,
  "relevance": 1.0,
  "impact": 0.9,
  "timeliness": 0.9,
  "reasoning": "AlphaFold 3 represents a significant advancement over its predecessor by extending its capabilities to predict protein-ligand complex structures. This innovation is highly relevant to the AI research field, particularly in computational biology and drug discovery, as it addresses a complex problem with practical applications. The impact is substantial, as it could revolutionize drug discovery processes and inspire further research in protein structure prediction. The timeliness is high due to the ongoing importance of drug discovery and molecular biology in addressing global health challenges.",
  "overall_score": 0.93
}

Parsed Evaluation:
------------------------------------------------------------
Innovation: 0.9
Relevance: 1.0
Impact: 0.9
Timeliness: 0.9
Overall Score: 0.93

R

### 20. Category Classification with LLM

카테고리 분류 프롬프트 + LLM 통합 테스트

In [21]:
# 카테고리 분류 테스트
classification_data = {
    "title": "OpenAI Announces ChatGPT Enterprise",
    "content": """OpenAI today announced ChatGPT Enterprise, bringing advanced AI 
    capabilities to businesses. The new offering includes enhanced security, 
    unlimited high-speed GPT-4 access, and custom model fine-tuning.""",
    "source_name": "TechCrunch",
    "url": "https://techcrunch.com/example"
}

print("Category Classification with LLM:")
print("=" * 60)

messages = manager.build_messages(
    "classify_category",
    **classification_data
)

classification_result = llm_client.chat_completion(
    messages,
    response_format="json",
    max_tokens=400
)

print("Classification Result:")
print("-" * 60)
print(classification_result)

# JSON 파싱
try:
    class_data = json.loads(classification_result)
    print("\nParsed Classification:")
    print("-" * 60)
    print(f"Category: {class_data.get('category', 'N/A')}")
    print(f"Confidence: {class_data.get('confidence', 'N/A')}")
    print(f"Keywords: {class_data.get('keywords', [])}")
    print(f"Research Field: {class_data.get('research_field', 'N/A')}")
    print(f"Sub-fields: {class_data.get('sub_fields', [])}")
    print(f"\nReasoning: {class_data.get('reasoning', 'N/A')}")
except json.JSONDecodeError as e:
    print(f"✗ JSON parsing failed: {e}")

Category Classification with LLM:
Classification Result:
------------------------------------------------------------
{
  "category": "news",
  "confidence": 0.95,
  "keywords": ["OpenAI", "ChatGPT Enterprise", "AI capabilities", "businesses", "GPT-4"],
  "research_field": "Artificial Intelligence",
  "sub_fields": ["Natural Language Processing", "Business Applications"],
  "reasoning": "The content is an announcement about a new product from OpenAI, reported by TechCrunch, a known tech news outlet. This fits the definition of a news article, particularly focusing on AI applications in business."
}

Parsed Classification:
------------------------------------------------------------
Category: news
Confidence: 0.95
Keywords: ['OpenAI', 'ChatGPT Enterprise', 'AI capabilities', 'businesses', 'GPT-4']
Research Field: Artificial Intelligence
Sub-fields: ['Natural Language Processing', 'Business Applications']

Reasoning: The content is an announcement about a new product from OpenAI, reporte

### 21. Batch Processing Test

여러 프롬프트를 동시에 처리

In [22]:
import asyncio

async def batch_process_article(article_data):
    """단일 기사에 대해 요약, 평가, 분류를 동시에 수행"""
    client = LLMClient(provider="openai", temperature=0.5)
    
    # 1. 요약
    summary_messages = manager.build_messages(
        "summarize",
        "korean.medium",
        title=article_data["title"],
        content=article_data["content"]
    )
    
    # 2. 중요도 평가
    eval_messages = manager.build_messages(
        "evaluate_importance",
        title=article_data["title"],
        content=article_data["content"],
        metadata=json.dumps(article_data.get("metadata", {}))
    )
    
    # 3. 카테고리 분류
    class_messages = manager.build_messages(
        "classify_category",
        title=article_data["title"],
        content=article_data["content"],
        source_name=article_data.get("source_name", "Unknown"),
        url=article_data.get("url", "")
    )
    
    # 동시 실행
    summary, evaluation, classification = await asyncio.gather(
        client.achat_completion(summary_messages, max_tokens=300),
        client.achat_completion(eval_messages, response_format="json", max_tokens=400),
        client.achat_completion(class_messages, response_format="json", max_tokens=400)
    )
    
    return {
        "summary": summary,
        "evaluation": evaluation,
        "classification": classification
    }

# 테스트 실행
test_article_data = {
    "title": "Gemini 1.5 Pro: Next Generation Multimodal AI",
    "content": """Google DeepMind introduces Gemini 1.5 Pro with a breakthrough 
    1 million token context window. The model demonstrates exceptional performance 
    on long-context tasks and multimodal understanding.""",
    "source_name": "Google AI Blog",
    "url": "https://ai.googleblog.com/example",
    "metadata": {"date": "2024-12-14", "author": "Google DeepMind"}
}

print("Batch Processing Test:")
print("=" * 60)

result = await batch_process_article(test_article_data)

print("\nSummary:")
print("-" * 60)
print(result["summary"])

print("\nEvaluation:")
print("-" * 60)
print(result["evaluation"])

print("\nClassification:")
print("-" * 60)
print(result["classification"])

Batch Processing Test:

Summary:
------------------------------------------------------------
Google DeepMind가 발표한 Gemini 1.5 Pro는 100만 개의 토큰 문맥 창을 갖춘 차세대 멀티모달 AI 모델입니다. 이 모델은 긴 문맥을 필요로 하는 작업과 멀티모달 이해에서 뛰어난 성능을 보입니다. 이러한 혁신은 AI의 응용 가능성을 크게 확장할 것으로 기대됩니다.

Evaluation:
------------------------------------------------------------
{
  "innovation": 0.8,
  "relevance": 0.9,
  "impact": 0.85,
  "timeliness": 0.9,
  "reasoning": "Gemini 1.5 Pro introduces a significant advancement in AI with a 1 million token context window, which is a notable improvement over existing models. This enhances its capabilities in long-context tasks and multimodal understanding, areas that are highly relevant in the current AI landscape. The innovation lies in the ability to handle large context windows, which can influence both academic research and practical applications. The relevance and timeliness are high due to the ongoing trend of improving AI models' contextual understanding and multimodal capabilities, 

### Summary

이 노트북에서 다룬 내용:

#### Basic Tests (1-7)
1. ✓ PromptManager 초기화
2. ✓ 카테고리 목록 확인
3. ✓ 요약 길이 옵션
4. ✓ 특정 프롬프트 가져오기
5. ✓ 유저 템플릿 가져오기
6. ✓ 프롬프트 변수 치환
7. ✓ LLM 메시지 빌드

#### Prompt Categories (8-12)
8. ✓ 모든 요약 길이 테스트
9. ✓ 중요도 평가 프롬프트
10. ✓ 카테고리 분류 프롬프트
11. ✓ 메타데이터 추출 프롬프트
12. ✓ 온보딩 프롬프트

#### Advanced Features (13-17)
13. ✓ 공통 설정 확인
14. ✓ 싱글톤 패턴 테스트
15. ✓ 편의 함수 테스트
16. ✓ 프롬프트 재로드
17. ✓ 존재하지 않는 키 처리

#### Integration Tests (18-21)
18. ✓ LLM 통합 테스트 (요약)
19. ✓ 중요도 평가 + LLM
20. ✓ 카테고리 분류 + LLM
21. ✓ 배치 처리 (요약 + 평가 + 분류)

PromptManager가 정상적으로 작동하며, 다양한 LLM 작업에 활용할 수 있습니다.